In [1]:
import pandas as pd
listings=pd.read_csv("listings.csv.gz")

In [2]:
listings[
[
    "price",
    "estimated_revenue_l365d",
    "estimated_occupancy_l365d",
    "review_scores_rating"
]
].isnull().sum()

price                        34908
estimated_revenue_l365d      34908
estimated_occupancy_l365d        0
review_scores_rating         24122
dtype: int64

In [3]:
listings.isnull().sum().sort_values(ascending=False).head(20)

neighbourhood_group_cleansed    96871
calendar_updated                96871
license                         96871
neighborhood_overview           55663
neighbourhood                   55662
host_neighbourhood              51021
host_about                      47038
beds                            34920
estimated_revenue_l365d         34908
price                           34908
bathrooms                       34846
host_response_time              31707
host_response_rate              31707
host_acceptance_rate            27760
review_scores_value             24166
review_scores_location          24166
review_scores_checkin           24165
review_scores_communication     24142
review_scores_accuracy          24137
review_scores_cleanliness       24131
dtype: int64

In [4]:
cols_to_drop=[
    "neighbourhood_group_cleansed",
    "calendar_updated",
    "license"
]
listings.drop(columns=cols_to_drop,inplace=True)

In [6]:
listings["price"]=(
    listings["price"]
    .str.replace("$","",regex=False)
    .str.replace(",","",regex=False)
)
listings["price"]=pd.to_numeric(
    listings["price"],
    errors="coerce"
)

In [7]:
listings["price"].dtype

dtype('float64')

In [9]:
listings[
    listings["price"].isna()
][
[
    "host_name",
    "room_type",
    "estimated_revenue_l365d",
    "estimated_occupancy_l365d"
]
].head()

,host_name,room_type,estimated_revenue_l365d,estimated_occupancy_l365d
3,Joe,Entire home/apt,NaN,14
10,Alec,Private room,NaN,0
12,Peter,Private room,NaN,0
14,Bimpe,Private room,NaN,0
16,Shelley,Private room,NaN,0


In [10]:
listings["price"].isna().mean()*100

36.03555243571348

In [11]:
listings=listings[listings["price"].notna()]

In [12]:
listings.shape

(61963, 76)

In [13]:
listings=listings[listings["estimated_revenue_l365d"].notna()]

In [14]:
listings["review_scores_rating"]=(
    listings["review_scores_rating"].fillna(listings["review_scores_rating"].median()))

In [17]:
listings["review_scores_rating"].isnull().sum()

0

In [18]:
listings.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type',
       'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms',
       'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights',
       'minimum_minimum_nights', 'maximum_minimum_nights',
       'minimum_maximum_nights', 'maximum_maximum_nights',
       'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability',
       'availability_3

In [47]:
listings = listings.copy()

In [19]:
keep_cols=[
    "id",
    "name",
    "host_name",
    "host_since",
    "host_response_rate",
    "host_acceptance_rate",
    "host_is_superhost",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "bathrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "amenities",
    "price",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "availability_365",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes"
]
listings=listings[keep_cols]

In [20]:
listings.shape

(61963, 37)

In [48]:
listings["amenity_count"]=(listings["amenities"].str.count(",") +1)

In [23]:
listings["amenity_count"].describe()

count    61963.000000
mean        31.428772
std         15.313782
min          1.000000
25%         19.000000
50%         32.000000
75%         42.000000
max        103.000000
Name: amenity_count, dtype: float64

In [49]:
listings["revenue_per_night"]=(
    listings["estimated_revenue_l365d"]/listings["estimated_occupancy_l365d"])

In [42]:
listings[listings["estimated_occupancy_l365d"] == 0][
[
    "price",
    "estimated_revenue_l365d",
    "estimated_occupancy_l365d"
]
].head()

,price,estimated_revenue_l365d,estimated_occupancy_l365d
2,411.0,0.0,0
7,61.0,0.0,0
8,340.0,0.0,0
11,213.0,0.0,0
19,76.0,0.0,0


In [43]:
listings[
    listings["estimated_occupancy_l365d"] == 0
]["estimated_revenue_l365d"].describe()

count    20966.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: estimated_revenue_l365d, dtype: float64

In [44]:
(
    (listings["estimated_occupancy_l365d"] == 0)
    &
    (listings["estimated_revenue_l365d"] == 0)
).sum()

20966

In [45]:
listings = listings[
    ~(
        (listings["estimated_occupancy_l365d"] == 0)
        &
        (listings["estimated_revenue_l365d"] == 0)
    )
].copy()

In [46]:
listings["revenue_per_night"].describe()

count    40889.000000
mean       168.221453
std        164.770740
min         10.000000
25%         75.000000
50%        129.000000
75%        204.000000
max       5000.000000
Name: revenue_per_night, dtype: float64

In [26]:
listings.duplicated().sum()

0

In [27]:
listings["price"].describe()

count    6.196300e+04
mean     2.299170e+02
std      4.437589e+03
min      7.000000e+00
25%      7.700000e+01
50%      1.350000e+02
75%      2.210000e+02
max      1.085147e+06
Name: price, dtype: float64

In [28]:
listings["estimated_revenue_l365d"].describe()

count    6.196300e+04
mean     1.047348e+04
std      6.612429e+04
min      0.000000e+00
25%      0.000000e+00
50%      2.968000e+03
75%      1.225000e+04
max      1.015000e+07
Name: estimated_revenue_l365d, dtype: float64

In [29]:
listings.sort_values(
    "estimated_revenue_l365d",
    ascending=False
).head(20)

,id,name,host_name,host_since,host_response_rate,host_acceptance_rate,host_is_superhost,neighbourhood_cleansed,latitude,longitude,...,review_scores_communication,review_scores_location,review_scores_value,availability_365,estimated_occupancy_l365d,estimated_revenue_l365d,calculated_host_listings_count,calculated_host_listings_count_entire_homes,amenity_count,revenue_per_night
57204,1009450636308513568,Close to London Eye (BOL),Wilson,2022-02-08,99%,98%,f,Lambeth,51.496114,-0.114789,...,4.72,4.70,4.39,269,175,10150000.0,33,0,14,58000.0
51162,904417963453083510,Twin Room Close to London Eye (RHI),Wilson,2022-02-08,99%,98%,f,Lambeth,51.494286,-0.112889,...,4.76,4.81,4.55,357,202,10100000.0,33,0,14,50000.0
69658,1210192079450300815,Walk To London Eye (BLZ),Wilson,2022-02-08,99%,98%,f,Lambeth,51.496466,-0.114816,...,4.67,4.78,4.44,365,129,3870000.0,33,0,6,30000.0
75312,1289793379859667497,Very Central Room - Walk to Eye,Eddy,2017-08-07,100%,99%,f,Lambeth,51.494530,-0.114569,...,4.40,4.80,4.20,357,46,2668000.0,28,1,5,58000.0
55413,974784336943790669,Stylish Earl's Court Getaway,Maxime,2023-09-06,100%,99%,t,Kensington and Chelsea,51.492180,-0.192930,...,4.92,4.91,4.77,200,255,2549745.0,1,1,23,9999.0
77023,1311151886101957046,Walk To London Eye,Eddy,2017-08-07,100%,99%,f,Lambeth,51.494578,-0.113249,...,4.00,4.33,3.33,357,28,1853292.0,28,1,5,66189.0
69137,1203041287118705621,Short Walk to London Eye (TOK),Wilson,2022-02-08,99%,98%,f,Lambeth,51.496288,-0.114585,...,4.64,4.55,3.91,264,64,1792000.0,33,0,5,28000.0
71302,1234699764935701963,Tranquil 2 Bedroom Apartment in East London,Maxime,2011-11-22,100%,99%,f,Tower Hamlets,51.524686,-0.067181,...,4.74,4.58,4.11,330,175,1749825.0,252,252,28,9999.0
68167,1190791764087921511,Close to The Shard (TAI),Wilson,2022-02-08,99%,98%,f,Southwark,51.499098,-0.092622,...,4.46,4.50,4.13,365,147,1470147.0,33,0,13,10001.0
31553,41559554,Close To London Eye (TUR),Eddy,2017-08-07,100%,99%,f,Lambeth,51.494550,-0.114030,...,4.95,4.50,4.26,357,212,1413192.0,28,1,8,6666.0


In [33]:
(listings["price"] > 5000).sum()

105

In [34]:
(listings["estimated_revenue_l365d"] > 500000).sum()

15

In [37]:
listings = listings[
    listings["price"] <= 5000
]

listings = listings[
    listings["estimated_revenue_l365d"] <= 500000
]

In [38]:
listings.shape

(61855, 39)

In [39]:
listings["price"].describe()

count    61855.000000
mean       189.523337
std        234.045816
min          7.000000
25%         77.000000
50%        135.000000
75%        220.000000
max       5000.000000
Name: price, dtype: float64

In [40]:
listings["estimated_revenue_l365d"].describe()

count     61855.000000
mean       9777.791221
std       17712.570800
min           0.000000
25%           0.000000
50%        2970.000000
75%       12250.000000
max      418200.000000
Name: estimated_revenue_l365d, dtype: float64

In [50]:
listings["host_since"]=pd.to_datetime(listings["host_since"])
listings["host_experience_years"]=(pd.Timestamp.today() - listings["host_since"]).dt.days/365

In [51]:
listings.shape

(40889, 40)

In [52]:
listings.isnull().sum().sort_values(
    ascending=False
).head(20)

host_response_rate                1689
host_is_superhost                 1063
host_acceptance_rate               562
bedrooms                            70
beds                                22
bathrooms                           19
host_experience_years               11
host_name                           11
host_since                          11
availability_365                     0
review_scores_accuracy               0
review_scores_cleanliness            0
review_scores_checkin                0
review_scores_communication          0
review_scores_location               0
review_scores_value                  0
estimated_occupancy_l365d            0
availability_365                     0
estimated_revenue_l365d              0
calculated_host_listings_count       0
dtype: int64

In [53]:
listings.head()

,id,name,host_name,host_since,host_response_rate,host_acceptance_rate,host_is_superhost,neighbourhood_cleansed,latitude,longitude,...,review_scores_location,review_scores_value,availability_365,estimated_occupancy_l365d,estimated_revenue_l365d,calculated_host_listings_count,calculated_host_listings_count_entire_homes,amenity_count,revenue_per_night,host_experience_years
0,13913,Holiday London DB Room Let-on going,Alina,2009-11-16,100%,96%,t,Islington,51.56861,-0.11270,...,4.78,4.78,331,92,6440.0,2,1,55,70.0,16.572603
1,15400,Bright Chelsea Apartment. Chelsea!,Philippa,2009-12-05,NaN,50%,f,Kensington and Chelsea,51.48780,-0.16813,...,4.93,4.74,199,9,1341.0,1,1,25,149.0,16.520548
4,36274,Bright 1 bedroom apt off brick lane in Shoreditch,Hendryks,2010-05-27,100%,98%,f,Tower Hamlets,51.52322,-0.06979,...,4.85,4.54,323,60,12600.0,2,2,6,210.0,16.046575
5,36299,Kew Gardens 3BR house in cul-de-sac,Geert,2010-06-30,100%,91%,f,Richmond upon Thames,51.48145,-0.28107,...,4.90,4.65,324,55,15400.0,1,1,34,280.0,15.953425
6,36660,You are GUARANTEED to love this,Agri & Roger,2010-07-04,100%,100%,t,Haringey,51.58478,-0.16057,...,4.77,4.86,289,255,22950.0,2,0,26,90.0,15.942466


In [54]:
listings.to_csv("clean_listings.csv",index=False)